# 03 Comparable Bond Selection

This notebook ranks peer bonds for each target using simple, explainable similarity rules and uses those peers to infer a fair spread.

The point of this step is not to claim that the scoring system is perfect. It is to show a clear and defendable process for identifying relevant peers, weighting them, and turning that peer set into a fair-value anchor.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd

from src.comps import infer_fair_spreads, select_top_comps

In [3]:
analytics_path = PROJECT_ROOT / "outputs" / "bond_analytics.csv"
universe_path = PROJECT_ROOT / "data" / "raw" / "bond_universe.csv"

analytics = pd.read_csv(analytics_path, parse_dates=["evaluation_date", "maturity_date"])
universe = pd.read_csv(universe_path, parse_dates=["evaluation_date", "maturity_date"])
bonds = analytics.merge(
    universe,
    on=["bond_id", "issuer", "evaluation_date", "maturity_date", "coupon_rate", "clean_price"],
    how="left",
)
bonds[["bond_id", "issuer", "rating", "sector", "spread_to_curve"]]

,bond_id,issuer,rating,sector,spread_to_curve
0,GE_1994_650,GE,AA,Industrials,-0.007052
1,GE_1998_725,GE,AA,Industrials,-0.005841
2,IBM_1993_625,IBM,AA,Technology,-0.009434
3,IBM_1996_700,IBM,AA,Technology,-0.009468
4,JNJ_1995_675,JNJ,AAA,Healthcare,-0.008968
5,JNJ_2000_740,JNJ,AAA,Healthcare,-0.005196
6,XOM_1997_710,XOM,AA,Energy,-0.006754
7,XOM_2001_760,XOM,AA,Energy,-0.001546


In [4]:
top_comps = select_top_comps(bonds, max_comps=3)
top_comps

,target_bond_id,comp_bond_id,target_issuer,comp_issuer,issuer_match,rating_match,seniority_match,sector_match,currency_match,maturity_gap_years,coupon_gap_pct,spread_gap_bps,comp_score,comp_weight
0,GE_1994_650,GE_1998_725,GE,GE,1,1,1,1,1,4.709103,0.75,12.103046,4.994626,4.994626
1,GE_1994_650,IBM_1993_625,GE,IBM,0,1,1,0,1,0.709103,0.25,23.824343,3.092987,3.092987
2,GE_1994_650,XOM_1997_710,GE,XOM,0,1,1,0,1,2.962355,0.60,2.978707,1.740755,1.740755
3,GE_1998_725,GE_1994_650,GE,GE,1,1,1,1,1,4.709103,0.75,12.103046,4.994626,4.994626
4,GE_1998_725,XOM_1997_710,GE,XOM,0,1,1,0,1,1.746749,0.15,9.124339,2.753871,2.753871
5,GE_1998_725,IBM_1996_700,GE,IBM,0,1,1,0,1,2.165640,0.25,36.270475,1.554374,1.554374
6,IBM_1993_625,IBM_1996_700,IBM,IBM,1,1,1,1,1,3.252567,0.75,0.343086,6.512654,6.512654
7,IBM_1993_625,GE_1994_650,IBM,GE,0,1,1,0,1,0.709103,0.25,23.824343,3.092987,3.092987
8,IBM_1993_625,JNJ_1995_675,IBM,JNJ,0,0,1,0,1,2.127310,0.50,4.659516,0.408366,0.408366
9,IBM_1996_700,IBM_1993_625,IBM,IBM,1,1,1,1,1,3.252567,0.75,0.343086,6.512654,6.512654


In [5]:
fair_spreads = infer_fair_spreads(bonds, top_comps)
fair_spreads

,bond_id,fair_spread,comp_count,comp_support_score,comp_ids
0,GE_1994_650,-0.007134,3,3.276123,"GE_1998_725, IBM_1993_625, XOM_1997_710"
1,GE_1998_725,-0.007367,3,3.100957,"GE_1994_650, XOM_1997_710, IBM_1996_700"
2,IBM_1993_625,-0.008702,3,3.338003,"IBM_1996_700, GE_1994_650, JNJ_1995_675"
3,IBM_1996_700,-0.008165,3,3.789177,"IBM_1993_625, XOM_1997_710, GE_1998_725"
4,JNJ_1995_675,-0.006308,3,2.067624,"JNJ_2000_740, IBM_1996_700, GE_1994_650"
5,JNJ_2000_740,-0.008225,3,1.771197,"JNJ_1995_675, GE_1998_725, XOM_2001_760"
6,XOM_1997_710,-0.005379,3,3.303057,"XOM_2001_760, IBM_1996_700, GE_1998_725"
7,XOM_2001_760,-0.006599,3,1.507743,"XOM_1997_710, GE_1998_725, JNJ_2000_740"


In [6]:
comp_view = top_comps[["target_bond_id", "comp_bond_id", "comp_score", "maturity_gap_years", "spread_gap_bps"]].copy()
comp_view.sort_values(["target_bond_id", "comp_score"], ascending=[True, False])

,target_bond_id,comp_bond_id,comp_score,maturity_gap_years,spread_gap_bps
0,GE_1994_650,GE_1998_725,4.994626,4.709103,12.103046
1,GE_1994_650,IBM_1993_625,3.092987,0.709103,23.824343
2,GE_1994_650,XOM_1997_710,1.740755,2.962355,2.978707
3,GE_1998_725,GE_1994_650,4.994626,4.709103,12.103046
4,GE_1998_725,XOM_1997_710,2.753871,1.746749,9.124339
5,GE_1998_725,IBM_1996_700,1.554374,2.165640,36.270475
6,IBM_1993_625,IBM_1996_700,6.512654,3.252567,0.343086
7,IBM_1993_625,GE_1994_650,3.092987,0.709103,23.824343
8,IBM_1993_625,JNJ_1995_675,0.408366,2.127310,4.659516
9,IBM_1996_700,IBM_1993_625,6.512654,3.252567,0.343086


In a production setting, comp selection would likely bring in more issuer history, sector conventions, liquidity signals, and trade evidence. For this project, the simpler rule set is useful because every part of the decision can be inspected directly.